# Business Understanding

In the modern digital era (circa 2026), the rapid expansion of machine learning has significantly influenced how surveillance is conducted.This project is inspired by the legal precedent established in Terry v. Ohio, which introduced the concept of “reasonable suspicion” in stop-and-frisk practices. While originally rooted in human judgment, similar decision-making processes are now being translated into algorithmic systems. This raises important questions about how historical data, when used to train machine learning models, may shape modern surveillance outcomes.

## Business Objective
The primary objective of this project is to explore the impact of using historically grounded policing datasets to train machine learning models in a digital surveillance context. 

## Stakeholders
The key stakeholders in this context include:

1.Law Enforcement Agencies – Seeking to improve efficiency and consistency in decision-making through data-driven tools.

2.Government & Policymakers – Responsible for regulating surveillance technologies and ensuring alignment with legal frameworks.

3.Technology Companies – Building and deploying machine learning systems used in surveillance and analytics.

4.Civil Rights Organizations – Evaluating fairness, accountability, and potential discrimination in algorithmic systems.

5.General Public – Individuals who may be directly or indirectly affected by automated surveillance decisions.

## Business Problem
The project focuses on :

​. Pattern Identification: Analyzing variables used to justify "reasonable suspicion" decisions .

. ​Predictive Modeling: Training classifiers to predict stop outcomes based on observed variables like officer gender ,subject perceived race  and stop location


# Data Understanding
​The dataset used for this project is the Terry Stops dataset, provided by the City of Seattle via the Seattle Police Department (SPD) Open Data portal. This dataset contains records of police-reported stops under the legal precedent of Terry v. Ohio. Each of the 66,800+ rows represents a unique stop, providing a granular look at law enforcement interactions in Seattle from 2017 through early 2026.

## ​Data Source
​Publisher: Seattle Police Department
​Update Frequency: Daily (last updated March 16, 2026)
​Scope: Records include perceived demographics of the subject, officer demographics, and specific call details from the Computer-Aided Dispatch (CAD) system.

## Target Variable

​To address the business problem of predicting stop outcomes and identifying patterns in "reasonable suspicion," the primary target for our machine learning model is:

​Stop Resolution: This categorical feature describes the final outcome of the stop (e.g., Arrest, Field Contact, Offense Report, or no action taken). Predicting this allows us to understand which variables most strongly correlate with high-stakes outcomes like arrests.

## ​Predictor Variables (Features)

​The dataset provides 23 columns that serve as predictors to help the model identify patterns. Key features include:

​Subject Demographics: Subject Age Group, Subject Perceived Race, and Subject Perceived Gender.

​Officer Demographics: Officer Race, Officer Gender, and Officer YOB (Year of Birth/Experience).

​Contextual/Environmental Data: Precinct, Sector, Beat, and Occurred Date (to capture time-based trends).

​Stop Specifics: Weapon Type (if any), Arrest Flag, Frisk Flag, and Initial Call Type (why the stop was initiated).

​Note on Data Ethics: Because this data relies on "perceived" demographics reported by officers, the analysis focuses on the impact of these perceptions and potential biases within the recorded data rather than an objective demographic census.

# Data Preparation

In [193]:
# import the relevant libraries
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier 
from sklearn.metrics import accuracy_score, roc_curve, auc,classification_report,confusion_matrix
from sklearn import tree 
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer

df = pd.read_csv('Terry_Stops.csv')
df.head()

,Subject Age Group,Subject ID,GO / SC Num,Terry Stop ID,Stop Resolution,Weapon Type,Officer ID,Officer YOB,Officer Gender,Officer Race,Subject Perceived Race,Subject Perceived Gender,Reported Date,Initial Call Type,Final Call Type,Call Type,Officer Squad,Arrest Flag,Frisk Flag,Precinct,Sector,Beat,Occurred Date
0,46 - 55,7729078044,2.023000e+13,51923259552,Field Contact,-,4494,1961,Male,White,Asian,Male,2023-08-16T23:45:14,DISTURBANCE,MISCHIEF OR NUISANCE - GENERAL,911,CRG - SQUAD 81D,N,Y,Southwest,W,W1,2023-08-16T20:31:00
1,36 - 45,7727706299,2.025000e+13,63109655222,Field Contact,-,8974,1997,Female,White,White,Male,2025-03-08T18:29:04,SUSPICIOUS STOP - OFFICER INITIATED ONVIEW,SUSPICIOUS CIRCUM. - SUSPICIOUS PERSON,ONVIEW,NORTH PCT 2ND W - BOY (JOHN) - PLATOON 1,N,N,North,U,U2,2025-03-08T17:58:00
2,-,31629429379,2.022000e+13,31629401025,Field Contact,Knife/Cutting/Stabbing Instrument,6885,1976,Male,Asian,White,Female,2022-02-15T18:37:10,DISTURBANCE,SUSPICIOUS CIRCUM. - SUSPICIOUS PERSON,ONVIEW,WEST PCT 2ND W - SPECIAL BEATS,N,Y,West,M,M2,2022-02-15T17:42:00
3,46 - 55,7746702884,2.021000e+13,25601408632,Arrest,-,8696,1996,Male,White,White,Male,2021-06-18T01:57:37,SUSPICIOUS STOP - OFFICER INITIATED ONVIEW,WARRANT SERVICES - MISDEMEANOR,ONVIEW,NORTH PCT 3RD W - B/N RELIEF,Y,N,North,N,N2,2021-06-18T00:48:00
4,17-Jan,-1,2.017000e+13,301638,Arrest,NaN,7773,1978,Male,White,Black or African American,Male,2017-08-27T03:18:00,OBS - FIGHT - IP - PHYSICAL (NO WEAPONS),"ASSAULTS, OTHER",911,NORTH PCT 3RD W - B/N RELIEF,N,N,North,L,L3,2017-08-27T03:18:00


In [194]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 66786 entries, 0 to 66785
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Subject Age Group         66786 non-null  object 
 1   Subject ID                66786 non-null  int64  
 2   GO / SC Num               66786 non-null  float64
 3   Terry Stop ID             66786 non-null  int64  
 4   Stop Resolution           66786 non-null  object 
 5   Weapon Type               34221 non-null  object 
 6   Officer ID                66786 non-null  object 
 7   Officer YOB               66786 non-null  int64  
 8   Officer Gender            66786 non-null  object 
 9   Officer Race              66786 non-null  object 
 10  Subject Perceived Race    66786 non-null  object 
 11  Subject Perceived Gender  66786 non-null  object 
 12  Reported Date             66786 non-null  object 
 13  Initial Call Type         66786 non-null  object 
 14  Final 

In [195]:
# drop unnecessary columns
cols_to_drop = ['Subject ID','GO / SC Num','Terry Stop ID','Officer ID','Reported Date','Officer Squad','Precinct','Sector','Beat']
df = df.drop(columns=cols_to_drop)

# create binary target
df['target'] = df['Stop Resolution'].map(lambda x : 1 if x == 'Arrest' else 0)
no_weapon = ['-','None','NaN']
df['weapon_binary'] = df['Weapon Type'].apply(lambda x : 0 if (str(x) in no_weapon or pd.isna(x))else 1 )

#  First, fix the 'Date' error in the Age Group
df['Subject Age Group'] = df['Subject Age Group'].replace('17-Jan', '1 - 17')

# identify all categorical columns you are keeping
categorical_cols = df.select_dtypes(include=['object']).columns

# Fill any remaining NaNs with 'Unknown' 
df[categorical_cols] = df[categorical_cols].fillna('Unknown')

# 4. Check one last time - everything should be 66786
print(df.isnull().sum())

Subject Age Group           0
Stop Resolution             0
Weapon Type                 0
Officer YOB                 0
Officer Gender              0
Officer Race                0
Subject Perceived Race      0
Subject Perceived Gender    0
Initial Call Type           0
Final Call Type             0
Call Type                   0
Arrest Flag                 0
Frisk Flag                  0
Occurred Date               0
target                      0
weapon_binary               0
dtype: int64


In [196]:
# 1. Convert the column to a datetime object
df['Occurred Date'] = pd.to_datetime(df['Occurred Date'])

# 2. Extract the hour (0-23) into a new numerical column
df['Stop_Hour'] = df['Occurred Date'].dt.hour

# 3. Drop the original date column since the model can't read it
df = df.drop(columns=['Occurred Date'])

In [197]:
X = df.drop(columns=['Stop Resolution','target','Weapon Type'])
y = df['target']

X_train , X_test ,y_train , y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [198]:
cat_cols = ['Subject Age Group', 'Officer Gender', 'Officer Race','Subject Perceived Race', 'Subject Perceived Gender',
            'Initial Call Type', 'Final Call Type', 'Call Type', 'Arrest Flag','Frisk Flag']

num_col = ['Officer YOB','Stop_Hour']

# 1. Initialize the transformer
preprocessor = ColumnTransformer(
    transformers=[
        # Apply OneHotEncoding to categorical features
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first'), cat_cols),
        
        # Apply Scaling to numerical features
        ('num', StandardScaler(), num_col)
    ],
    remainder='passthrough' # This keeps columns like 'weapon_binary' which are already 0/1
)

# 2. Fit and transform the training data
X_train_transformed = preprocessor.fit_transform(X_train)

# 3. Transform the test data 
X_test_transformed = preprocessor.transform(X_test)


c:\Users\HP PROBOOK 11X360G6\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:246: UserWarning: Found unknown categories in columns [5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [199]:
# Get the new column names created by the encoder
ohe_colnames = preprocessor.get_feature_names_out()

# Create a nice-looking DataFrame
X_train_final = pd.DataFrame(X_train_transformed, columns=ohe_colnames)

X_train_final.head()


,cat__Subject Age Group_1 - 17,cat__Subject Age Group_18 - 25,cat__Subject Age Group_26 - 35,cat__Subject Age Group_36 - 45,cat__Subject Age Group_46 - 55,cat__Subject Age Group_56 and Above,cat__Officer Gender_Male,cat__Officer Gender_Non-Specified,cat__Officer Race_Asian,cat__Officer Race_Black or African American,cat__Officer Race_Declined to Answer,cat__Officer Race_Hispanic,cat__Officer Race_Native Hawaiian or Other Pacific Islander,cat__Officer Race_Two or More Races,cat__Officer Race_Unknown,cat__Officer Race_White,cat__Subject Perceived Race_American Indian or Alaska Native,cat__Subject Perceived Race_Asian,cat__Subject Perceived Race_Black or African American,cat__Subject Perceived Race_Hispanic,cat__Subject Perceived Race_MULTIPLE SUBJECTS,cat__Subject Perceived Race_Multi-Racial,cat__Subject Perceived Race_Native Hawaiian or Other Pacific Islander,cat__Subject Perceived Race_Other,cat__Subject Perceived Race_Unknown,cat__Subject Perceived Race_White,cat__Subject Perceived Gender_Female,cat__Subject Perceived Gender_Gender Diverse (gender non-conforming and/or transgender),cat__Subject Perceived Gender_MULTIPLE SUBJECTS,cat__Subject Perceived Gender_Male,cat__Subject Perceived Gender_Unable to Determine,cat__Subject Perceived Gender_Unknown,cat__Initial Call Type_ABANDONED VEHICLE,cat__Initial Call Type_ABDUCTION - CRITICAL,cat__Initial Call Type_ABDUCTION - NO KNOWN KIDNAPPING,"cat__Initial Call Type_ALARM - ATM MACHINE, FREE STANDING",cat__Initial Call Type_ALARM - AUDIBLE AUTOMOBILE (UNOCC/ANTI-THEFT),cat__Initial Call Type_ALARM - BANK (HOLD-UP),"cat__Initial Call Type_ALARM - COMM, HOLD-UP/PANIC (EXCEPT BANKS)","cat__Initial Call Type_ALARM - COMM, SILENT/AUD BURG (INCL BANKS)",...,cat__Final Call Type_THREATS (INCLS IN-PERSON/BY PHONE/IN WRITING),cat__Final Call Type_TRAFFIC - ASSIST MOTORIST,cat__Final Call Type_TRAFFIC - BICYCLE VIOLATION,cat__Final Call Type_TRAFFIC - BLOCKING TRAFFIC,cat__Final Call Type_TRAFFIC - COMMUNITY TRAFFIC COMPLAINT (CTC),cat__Final Call Type_TRAFFIC - D.U.I.,cat__Final Call Type_TRAFFIC - MOVING VIOLATION,cat__Final Call Type_TRAFFIC - MV COLLISION INVESTIGATION,cat__Final Call Type_TRAFFIC - PARKING VIOL (EXCEPT ABANDONED CAR),cat__Final Call Type_TRAFFIC - PEDESTRIAN VIOLATION,cat__Final Call Type_TRAFFIC - REFUSE TO STOP (PURSUIT),cat__Final Call Type_TRAFFIC - TRAFFIC CONTROL (SPECIAL EVENTS),cat__Final Call Type_TRAFFIC STOP - OFFICER INITIATED ONVIEW,cat__Final Call Type_TRESPASS,cat__Final Call Type_UNKNOWN - COMPLAINT OF UNKNOWN NATURE,"cat__Final Call Type_URINATING, DEFECATING IN PUBLIC",cat__Final Call Type_VICE - GAMBLING,cat__Final Call Type_VICE - OTHER,cat__Final Call Type_VICE - PROSTITUTION,cat__Final Call Type_WARRANT - FELONY PICKUP,cat__Final Call Type_WARRANT - MISD PICKUP,cat__Final Call Type_WARRANT SERVICES - FELONY,cat__Final Call Type_WARRANT SERVICES - MISDEMEANOR,"cat__Final Call Type_WEAPN - GUN,DEADLY WPN (NO THRTS/ASLT/DIST)","cat__Final Call Type_WEAPON, PERSON WITH - GUN","cat__Final Call Type_WEAPON,PERSON WITH - OTHER WEAPON",cat__Call Type_911,cat__Call Type_ALARM CALL (NOT POLICE ALARM),cat__Call Type_HISTORY CALL (RETRO),cat__Call Type_ONVIEW,cat__Call Type_PROACTIVE (OFFICER INITIATED),cat__Call Type_SCHEDULED EVENT (RECURRING),"cat__Call Type_TELEPHONE OTHER, NOT 911",cat__Call Type_TEXT MESSAGE,cat__Arrest Flag_Y,cat__Frisk Flag_N,cat__Frisk Flag_Y,num__Officer YOB,num__Stop_Hour,remainder__weapon_binary
0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.897847,0.968144,0.0
1,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

# Modelling

## Model 1. Baseline logistic regression

In [200]:
logreg_base = LogisticRegression()
logreg_base.fit(X_train_final,y_train)

X_test_final = pd.DataFrame(X_test_transformed,columns=ohe_colnames)
y_pred_base = logreg_base.predict(X_test_final)


## Model 2. Tuned logistic regression (Balanced Weights)

When I changed the stop resolution column to binary it created class imbalance as there are more 'no arrest' than 'arrest'.This model attempts to fix that problem by telling the model to treat 'arrest' as more important than 'no arrest'.

In [201]:
df['target'].value_counts()

target
0    50861
1    15925
Name: count, dtype: int64

In [202]:
# Adding the 'balanced' hyperparameter to handle class imbalance
logreg_balanced = LogisticRegression( class_weight='balanced')
logreg_balanced.fit(X_train_final, y_train)
y_pred_bal = logreg_balanced.predict(X_test_final)

## Model 3. Decision Tree


In [203]:
dt_model = DecisionTreeClassifier(max_depth=5, class_weight='balanced',criterion='entropy')
dt_model.fit(X_train_final, y_train)
y_pred_tree = dt_model.predict(X_test_final)